# 10. Tables and figures of the paper

This notebook draws **Table 1, Figs. 1–4 and Table 2** of the paper from the aggregate results of notebooks 02–09 (it reads only `RESULTS_DIR`, never row-level data). Each item has its own section below, with its title and legend as in the paper. Tables are saved as CSV under `OUTPUT_DIR/tables`, figures as PNG (400 dpi) and PDF under `OUTPUT_DIR/figures`.

Results are given as ranges across the three models or, where stated, as their mean; "in most models" means that the 95% confidence interval excluded zero in at least two of the three models.

In [ ]:
%run ./00_settings.ipynb
from scipy.stats import norm, pearsonr, spearmanr

ROWS = [("Composite SHD", E_COMP, M_COMP), ("Aortic stenosis", E_AS, M_AS)]   # the two diagnoses of Figs. 2b, 3a and 3b
R = load_all()                                                                   # aggregate results (notebook 00, section 0.12)

## Table 1 | Characteristics of the two cohorts by recorded race and ethnicity

ECG, electrocardiogram; IQR, interquartile range; LVEF, left ventricular ejection fraction; SHD, structural heart disease.

Age, sex and prevalence refer to the most recent ECG of each patient. Composite SHD has 11 components in EchoNext and six in MIMIC-IV (Methods), so its prevalence is not comparable between cohorts. Patients recorded as other or unknown were not compared.

In [ ]:
ch = R["ch"].set_index(["cohort", "group"])
share = lambda v, tot: f"{int(v):,} ({100 * v / tot:.1f})"
rows = []
for c, role in (("EchoNext", "primary cohort"), ("MIMIC-IV", "replication cohort")):
    rows.append([f"{c} ({role})"] + [""] * 7)
    tot = ch.loc[(c, "All")]
    for gname in ["white", "black", "hispanic", "asian", "other", "unknown", "All"]:
        r = ch.loc[(c, gname)]
        rows.append([GROUP_NAME[gname], share(r.patients, tot.patients), share(r.ecgs, tot.ecgs), r.age, f"{r.female_pct:.0f}", f"{r['prev_Composite SHD']:.1f}",
                     f"{r['prev_LVEF ≤45%']:.1f}", f"{r['prev_Aortic stenosis']:.1f} ({int(r['pos_Aortic stenosis']):,})"])
table1 = save_table(pd.DataFrame(rows, columns=["Group", "Patients, n (%)", "ECGs, n (%)", "Age, years, median (IQR)", "Female, %", "Composite SHD, %",
                                                "LVEF ≤45%, %", "Aortic stenosis, % (n)"]), "Table1")
table1

## Fig. 1 | Race and ethnicity are decodable from frozen ECG foundation model embeddings and are not explained by cardiac structure

**a**, Study design. **b**, Area under the receiver operating characteristic curve (AUROC) for separating Black from White patients with a linear probe on each pretrained model, on the same architecture with random weights, and with a model on ECG measurements, age and sex. **c**, The same AUROC among all patients, among patients without recorded structural heart disease (SHD), within pairs matched on cardiac structure, age, sex, clinical setting and calendar year (within-pair concordance), and after removal of structural components from the embeddings. The dashed line marks a model on structural variables alone. Horizontal lines show 95% confidence intervals. ECG, electrocardiogram.

In [ ]:
def figure1(R):
    ch = R["ch"].set_index(["cohort", "group"])
    row_b, row_c, top_a, gap_ab, gap_bc, bottom = 0.42, 0.47, 1.14, 0.66, 1.08, 0.72          # layout in inches
    hb, hc = row_b * (N_FM + 1) + 0.05, row_c * N_FM + 0.22
    H = 0.14 + top_a + gap_ab + hb + gap_bc + hc + bottom
    fig = plt.figure(figsize=(7.2, H))
    left, right, gx = 0.178, 0.988, 0.03
    w = (right - left - gx) / 2
    yb0, yc0 = 1 - (0.14 + top_a + gap_ab + hb) / H, bottom / H

    # ---- a: study design (cohorts, models, the three questions)
    a = fig.add_axes([0.012, 1 - (0.14 + top_a) / H, 0.976, top_a / H]); a.axis("off"); a.set_xlim(0, 1); a.set_ylim(0, 1)
    a.text(0.0, 1.05, "a", transform=a.transAxes, fontsize=10.5, fontweight="bold", color=INK, va="bottom", ha="left")
    T = a.transAxes

    def card(x, y, cw, chh, fill="#ffffff", edge=AXIS, lw=0.8):
        a.add_patch(FancyBboxPatch((x, y), cw, chh, boxstyle="round,pad=0,rounding_size=0.012", fc=fill, ec=edge, lw=lw, transform=T, clip_on=False,
                                   mutation_aspect=7.03 / top_a))

    def link(x0, y0, x1, y1, rad=0.0):
        a.annotate("", xy=(x1, y1), xytext=(x0, y0), xycoords=T, textcoords=T,
                   arrowprops=dict(arrowstyle="-|>", color=MUTED, lw=0.9, shrinkA=0, shrinkB=0, mutation_scale=8, connectionstyle=f"arc3,rad={rad}"))
    x1c, w1, x2c, w2, x3c = 0.0, 0.225, 0.268, 0.178, 0.482
    w3 = 1.0 - x3c
    for i, (c, role, col) in enumerate((("EchoNext", "primary cohort", CE), ("MIMIC-IV", "replication cohort", CM))):
        y0 = 0.54 - i * 0.54
        r = ch.loc[(c, "All")]
        card(x1c, y0, w1, 0.46)
        a.add_patch(FancyBboxPatch((x1c, y0 + 0.035), 0.0075, 0.39, boxstyle="round,pad=0,rounding_size=0.003", fc=col, ec=col, lw=0.8, transform=T,
                                   clip_on=False, mutation_aspect=7.03 / top_a))
        a.text(x1c + 0.02, y0 + 0.345, c, transform=T, fontsize=8.4, fontweight="bold", color=INK, va="center", ha="left")
        a.text(x1c + w1 - 0.012, y0 + 0.345, role, transform=T, fontsize=6.6, color=col, va="center", ha="right")
        a.text(x1c + 0.02, y0 + 0.205, f"{int(r.patients):,} patients", transform=T, fontsize=7.2, color=INK2, va="center", ha="left")
        a.text(x1c + 0.02, y0 + 0.085, f"{int(r.ecgs):,} ECGs", transform=T, fontsize=7.2, color=INK2, va="center", ha="left")
    card(x2c, 0.1, w2, 0.8, fill="#f4f7fb")
    for t, yy, kw in ((f"{N_WORD.capitalize()} frozen ECG", 0.735, dict(fontsize=8.3, fontweight="bold", color=INK)),
                      ("foundation models", 0.595, dict(fontsize=8.3, fontweight="bold", color=INK)),
                      ("one embedding per ECG", 0.395, dict(fontsize=7.1, color=INK2)), ("no fine-tuning", 0.265, dict(fontsize=7.1, color=INK2))):
        a.text(x2c + w2 / 2, yy, t, transform=T, va="center", ha="center", **kw)
    qs = [("Is race or ethnicity linearly decodable?", "linear probes, with random-weight and ECG-measurement comparators"),
          ("Is the signal explained by cardiac structure?", "restriction, matching, removal of structural components"),
          ("Do SHD detectors err differently across groups?", "common threshold, error rates by group, erasure of race information")]
    for i, (t, s) in enumerate(qs):
        y0 = 0.70 - i * 0.35
        card(x3c, y0, w3, 0.30, fill="#f4f7fb")
        a.text(x3c + 0.026, y0 + 0.15, str(i + 1), transform=T, fontsize=7.4, fontweight="bold", color="white", va="center", ha="center",
               bbox=dict(boxstyle="circle,pad=0.28", fc=INK2, ec="none"))
        a.text(x3c + 0.052, y0 + 0.205, t, transform=T, fontsize=8.0, fontweight="bold", color=INK, va="center", ha="left")
        a.text(x3c + 0.052, y0 + 0.085, s, transform=T, fontsize=6.9, color=INK2, va="center", ha="left")
        link(x2c + w2 + 0.004, 0.5, x3c - 0.005, y0 + 0.15, rad=(-0.16, 0.0, 0.16)[i])
    link(x1c + w1 + 0.004, 0.77, x2c - 0.005, 0.58, rad=-0.14)
    link(x1c + w1 + 0.004, 0.23, x2c - 0.005, 0.42, rad=0.14)

    # ---- b: race decodability (pretrained, random weights, ECG measurements)
    for j, (coh, s1) in enumerate((("EchoNext", R["s1e"]), ("MIMIC-IV", R["s1m"]))):
        ax = fig.add_axes([left + j * (w + gx), yb0, w, hb / H]); style(ax)
        col = COH[coh]
        ax.axvline(0.5, color=MUTED, lw=0.8, ls=":")
        separators(ax, N_FM + 1)
        for y, fm in enumerate(ORDER):
            r, rr = s1.loc[KEY[fm]], s1.loc[KEY[fm] + "_random"]
            ax.plot([r.bw_lo, r.bw_hi], [y - 0.17, y - 0.17], color=col, lw=1.6, solid_capstyle="round"); ax.plot(r.bw_auroc, y - 0.17, "o", color=col, ms=5.8)
            ax.plot([rr.bw_lo, rr.bw_hi], [y + 0.17, y + 0.17], color=MUTED, lw=1.2, solid_capstyle="round")
            ax.plot(rr.bw_auroc, y + 0.17, "o", color=MUTED, ms=5.3, mfc="white", mew=1.1)
        r = s1.loc["meas_lr"]
        ax.plot([r.bw_lo, r.bw_hi], [N_FM, N_FM], color=MUTED, lw=1.2); ax.plot(r.bw_auroc, N_FM, "s", color=MUTED, ms=5.0)
        ax.set_yticks(range(N_FM + 1)); ax.set_yticklabels(ORDER + ["ECG measurements"] if j == 0 else [], fontsize=8, color=INK)
        ax.set_ylim(N_FM + 0.5, -0.5); ax.set_xlim(0.48, 0.86); ax.set_xticks([0.5, 0.6, 0.7, 0.8])
        ax.set_xlabel("AUROC, Black vs White", fontsize=8, color=INK2)
        panel(ax, "b" if j == 0 else "", coh)
        if j == 0:
            ax.text(0.503, N_FM + 0.36, "chance", fontsize=6.8, color=MUTED, va="center")
            h = [Line2D([], [], color=INK2, marker="o", ms=5.8, lw=1.6, label="Pretrained foundation model"),
                 Line2D([], [], color=MUTED, marker="o", ms=5.3, mfc="white", mew=1.1, lw=1.2, label="Same architecture, random weights"),
                 Line2D([], [], color=MUTED, marker="s", ms=5.0, lw=1.2, label="ECG measurements, age and sex")]
            ax.legend(handles=h, loc="upper center", bbox_to_anchor=(1.02, -0.42 / hb), ncol=3, frameon=False, fontsize=7.4, handletextpad=0.5, columnspacing=1.4)

    # ---- c: structural adjustment (four values per model, each with its 95% CI)
    for j, (coh, S2, s1) in enumerate((("EchoNext", R["e2"], R["s1e"]), ("MIMIC-IV", R["m2"], R["s1m"]))):
        ax = fig.add_axes([left + j * (w + gx), yc0, w, hc / H]); style(ax)
        col = COH[coh]
        so = S2["m0"].loc["structure only", "bw_auroc"]
        ax.axvline(0.5, color=MUTED, lw=0.8, ls=":"); ax.axvline(so, color=INK2, lw=0.9, ls="--")
        ax.text(so - 0.005, -0.7, f"structure only {so:.2f}", fontsize=7, color=INK2, ha="right", va="center")
        separators(ax, N_FM)
        for y, fm in enumerate(ORDER):
            p, A_, B_ = s1.loc[KEY[fm]], S2["A"].loc[KEY[fm]], S2["B"].loc[KEY[fm]]
            c = S2["C"][(S2["C"].model == KEY[fm]) & (S2["C"].variant == "removed")].iloc[0]
            pts = [(p.bw_auroc, p.bw_lo, p.bw_hi, "o", -0.3), (A_.bw_auroc, A_.bw_lo, A_.bw_hi, "D", -0.1), (B_.pair_concordance, B_.lo, B_.hi, "^", 0.1),
                   (c.bw_auroc, c.bw_lo, c.bw_hi, "s", 0.3)]
            for v, lo, hi, mk, dy in pts:
                ax.plot([lo, hi], [y + dy, y + dy], color=col, lw=1.3, solid_capstyle="round", zorder=2)
                ax.plot(v, y + dy, mk, color=col, ms=6.2 if mk != "D" else 5.4, mec="white", mew=0.6, zorder=3)
        ax.set_yticks(range(N_FM)); ax.set_yticklabels(ORDER if j == 0 else [], fontsize=8, color=INK)
        ax.set_ylim(N_FM - 0.5, -1.0); ax.set_xlim(0.48, 0.86); ax.set_xticks([0.5, 0.6, 0.7, 0.8])
        ax.set_xlabel("AUROC, Black vs White", fontsize=8, color=INK2)
        panel(ax, "c" if j == 0 else "", coh)
        if j == 0:
            h = [Line2D([], [], color=INK2, marker=m, ms=6.2 if m != "D" else 5.4, lw=1.3, mec="white", mew=0.6, label=l) for m, l in
                 (("o", "All patients"), ("D", "No recorded SHD"), ("^", "Matched pairs (within-pair concordance)"), ("s", "Structural components removed"))]
            ax.legend(handles=h, loc="upper center", bbox_to_anchor=(1.02, -0.42 / hc), ncol=4, frameon=False, fontsize=7.4, handletextpad=0.4,
                      columnspacing=1.0, handlelength=1.6)
    save_figure(fig, "Fig1")


figure1(R)

## Fig. 2 | Two ways in which the errors of a detector differ between groups

**a**, Schematic of detector scores in patients with the diagnosis (above the axis, black line) and without it (below, grey line) at a common threshold (vertical line). Shaded areas show what each measure counts. The flagging rate is the share above the threshold, and balanced accuracy is the share on the correct side, each averaged over the two curves. A shifted group is flagged more often at nearly equal balanced accuracy, whereas a group whose patients with the diagnosis score lower is separated less well. Dashed lines repeat the reference group, and percentages are illustrative. **b**, Black–White difference in flagging rate and in balanced accuracy for composite structural heart disease (SHD) and aortic stenosis at the same age and sex, with 95% confidence intervals. **c**, Difference from White patients in both measures for every diagnosis, as the mean of the three models, with aortic stenosis circled. Small markers indicate fewer than 100 patients with the diagnosis.

In [ ]:
GAPY = 0.75
PAIR_NAMES = (("black-white", "Black"), ("hispanic-white", "Hispanic"), ("asian-white", "Asian"))
SK_S, SK_THR = 13.0, 50.0                                  # schematic of panel a: common SD of the score curves, and the threshold
DARK, LIGHT = "#8f8d86", "#d9d7cf"


def tmv(R, cohort, label_key, measure, pair="black-white", eraser="none"):
    d = R["tm"]
    return d[(d.cohort == cohort) & (d.label_key == label_key) & (d.measure == measure) & (d.pair == pair) & (d.eraser == eraser)].set_index("fm")


def row_y():
    ys, heads, y = {}, [], 0.0
    for b, (name, _, _) in enumerate(ROWS):
        heads.append((y - 0.78, name))
        for fm in ORDER:
            ys[(b, fm)] = y; y += 1
        y += GAPY
    return ys, heads, y - GAPY - 0.45


def block_axis(ax, j, xlim, step):
    ys, heads, ymax = row_y()
    style(ax)
    ax.axvline(0, color=INK2, lw=0.8)
    for (b, fm), y in ys.items():
        if fm != ORDER[-1]:
            ax.axhline(y + 0.5, color=GRID, lw=0.6, zorder=0)
    ax.set_yticks(list(ys.values())); ax.set_yticklabels([fm for (_, fm) in ys] if j == 0 else [], fontsize=8, color=INK)
    ax.set_ylim(ymax, -1.25); ax.set_xlim(*xlim); ax.xaxis.set_major_locator(MultipleLocator(step))
    if j == 0:
        for y, name in heads:
            ax.text(xlim[0] + 0.012 * (xlim[1] - xlim[0]), y, name, fontsize=8.2, fontweight="bold", color=INK, va="center", ha="left")
    return ys


def sketch_rates(m0, m1):
    """Sensitivity, false-positive rate, flagging rate and balanced accuracy of two normal score curves (means m0 without, m1 with the diagnosis)."""
    sens, fpr = 1 - norm.cdf(SK_THR, m1, SK_S), 1 - norm.cdf(SK_THR, m0, SK_S)
    return sens, fpr, (sens + fpr) / 2, (sens + 1 - fpr) / 2


def sketch(ax, m0, m1, row, ref, words):
    """Detector scores: with the diagnosis above the axis, without below; row 0 shades the flagging rate, row 1 the balanced accuracy."""
    x = np.linspace(0, 100, 600)
    top = norm.pdf(0, 0, SK_S) * 1.10
    yn, yp = norm.pdf(x, m0, SK_S), norm.pdf(x, m1, SK_S)
    if ref:
        ax.plot(x, norm.pdf(x, ref[1], SK_S), color=MUTED, lw=0.7, ls=(0, (3, 2))); ax.plot(x, -norm.pdf(x, ref[0], SK_S), color=MUTED, lw=0.7, ls=(0, (3, 2)))
    ax.fill_between(x, 0, yp, where=x >= SK_THR, color=DARK, lw=0)
    ax.fill_between(x, 0, -yn, where=(x >= SK_THR) if row == 0 else (x <= SK_THR), color=LIGHT, lw=0)
    ax.plot(x, yp, color=INK, lw=1.2); ax.plot(x, -yn, color=MUTED, lw=1.2)
    ax.axhline(0, color=AXIS, lw=0.8); ax.axvline(SK_THR, color=INK, lw=0.9)
    ax.set_xlim(6, 100); ax.set_ylim(-top, top); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    sens, fpr, _, _ = sketch_rates(m0, m1)
    ax.text(m1 + 3.5, top * 0.36, f"sensitivity\n{100 * sens:.0f}%" if words else f"{100 * sens:.0f}%", fontsize=6.4, color="white", ha="center", va="center",
            linespacing=1.05)
    if row == 0:
        ax.text(SK_THR + 2.0, -top * 0.82, f"false-positive rate {100 * fpr:.0f}%" if words else f"{100 * fpr:.0f}%", fontsize=6.4, color=INK2, ha="left", va="center")
    else:
        ax.text(m0 - 5.5, -top * 0.36, f"specificity\n{100 * (1 - fpr):.0f}%" if words else f"{100 * (1 - fpr):.0f}%", fontsize=6.4, color=INK2, ha="center",
                va="center", linespacing=1.05)
    return top


def figure2(R):
    H = 9.9
    fig = plt.figure(figsize=(7.2, H))
    # ---- a: what the two measures count (reference group, shifted operating point, poorer separation)
    ref = (SK_THR - norm.ppf(0.70) * SK_S, SK_THR + norm.ppf(0.80) * SK_S)          # reference group: 80% sensitivity, 30% false-positive rate
    scen = (("Reference group", ref, "one threshold for every group"),
            ("Shifted operating point", (ref[0] + 0.3 * SK_S, ref[1] + 0.3 * SK_S), "the whole group scores higher"),
            ("Poorer separation", (ref[0], ref[1] - 0.45 * SK_S), "patients with the diagnosis score lower"))
    x0, cw, gx, rh = 0.215, 0.243, 0.020, 0.95
    r0 = sketch_rates(*ref)
    for k, (title, (m0, m1), sub) in enumerate(scen):
        r = sketch_rates(m0, m1)
        for row in (0, 1):
            ax = fig.add_axes([x0 + k * (cw + gx), (H - 1.57 - row * 1.19) / H, cw, rh / H])
            top = sketch(ax, m0, m1, row, ref if k else None, k == 0)
            v, v0 = r[2 + row], r0[2 + row]
            ax.text(0.02, 0.97, f"{100 * v:.0f}%", transform=ax.transAxes, fontsize=10, fontweight="bold", color=INK, ha="left", va="top")
            if k:
                d = round(100 * v) - round(100 * v0)
                word = "nearly equal" if abs(d) <= 2 else ("higher" if d > 0 else "lower")
                ax.text(0.02, 0.78, f"{d:+d} points,\n{word}".replace("-", MINUS), transform=ax.transAxes, fontsize=6.4, color=INK2, ha="left", va="top", linespacing=1.12)
            if row == 0:
                ax.text(0, 1.22, title, transform=ax.transAxes, fontsize=8.4, fontweight="bold", color=INK, va="bottom", ha="left")
                ax.text(0, 1.07, sub, transform=ax.transAxes, fontsize=6.6, color=INK2, va="bottom", ha="left")
            if k == 0:
                if row == 0:
                    ax.text(SK_THR - 1.5, top * 0.97, "threshold", fontsize=6.2, color=INK, va="top", ha="right")
                    ax.text(99.5, top * 0.80, "with the\ndiagnosis", fontsize=6.2, color=INK, ha="right", va="center", linespacing=1.05)
                    ax.text(7, -top * 0.84, "without the\ndiagnosis", fontsize=6.2, color=MUTED, ha="left", va="center", linespacing=1.05)
                s, f = r[0], r[1]
                lab = ("Flagging rate", "share above the threshold,\nmean of the two curves", f"({100 * s:.0f}% + {100 * f:.0f}%) / 2 = {100 * (s + f) / 2:.0f}%") if row == 0 else (
                    "Balanced accuracy", "share on the correct side,\nmean of the two curves", f"({100 * s:.0f}% + {100 * (1 - f):.0f}%) / 2 = {100 * (s + 1 - f) / 2:.0f}%")
                ax.text(-0.86, 0.80, lab[0], transform=ax.transAxes, fontsize=8.4, fontweight="bold", color=INK, va="center", ha="left")
                ax.text(-0.86, 0.50, lab[1], transform=ax.transAxes, fontsize=6.6, color=INK2, va="center", ha="left", linespacing=1.2)
                ax.text(-0.86, 0.22, lab[2], transform=ax.transAxes, fontsize=6.6, color=MUTED, va="center", ha="left")
    fig.text(0.012, (H - 0.36) / H, "a", fontsize=10.5, fontweight="bold", color=INK)
    # ---- b: Black - White differences in the two measures, composite SHD and aortic stenosis
    yb, hb = (H - 5.83) / H, 2.45 / H
    left, right, gx = 0.178, 0.988, 0.03
    w = (right - left - gx) / 2
    for j, (meas, title, hint) in enumerate((("FLAG", "Flagging rate", "flagged more often"), ("BA", "Balanced accuracy", None))):
        ax = fig.add_axes([left + j * (w + gx), yb, w, hb])
        ys = block_axis(ax, j, (-15, 10), 5)
        for b, (_, e_lab, m_lab) in enumerate(ROWS):
            for coh, lab, mk, dy in (("EchoNext", e_lab, "o", -0.17), ("MIMIC-IV", m_lab, "s", 0.17)):
                d = tmv(R, coh, lab, meas)
                for fm in ORDER:
                    r, y = d.loc[fm], ys[(b, fm)] + dy
                    ax.plot([100 * r.lo, 100 * r.hi], [y, y], color=COH[coh], lw=1.5, solid_capstyle="round")
                    ax.plot(100 * r.value, y, mk, color=COH[coh], ms=5.2)
        ax.set_xlabel(f"Black − White difference in {title.lower()}\n(percentage points, same age and sex)", fontsize=7.8, color=INK2)
        panel(ax, "b" if j == 0 else "", title)
        hints(ax, "separated less well" if meas == "BA" else None, hint)
        if j == 1:
            h = [Line2D([], [], color=CE, marker="o", ms=5.2, lw=1.5, label="EchoNext"), Line2D([], [], color=CM, marker="s", ms=5.2, lw=1.5, label="MIMIC-IV")]
            ax.legend(handles=h, loc="upper right", bbox_to_anchor=(1.0, 0.93), frameon=False, fontsize=7.5, handletextpad=0.5, borderaxespad=0.2)
    # ---- c: every diagnosis, mean of the three models: difference in flagging rate (x) and balanced accuracy (y)
    T = R["tm"]
    T = T[(T.eraser == "none") & T.valid]
    pts, out = {}, {}
    for pair, _ in PAIR_NAMES:
        for coh in ("EchoNext", "MIMIC-IV"):
            d = T[(T.pair == pair) & (T.cohort == coh)]
            q = (d.pivot_table(index="label_key", columns="measure", values="value", aggfunc="mean") * 100).dropna(subset=["FLAG", "BA"])
            q["n"] = d.groupby("label_key").pos_a.first()
            q["sig"] = d[d.measure == "BA"].assign(s=lambda z: (z.lo > 0) | (z.hi < 0)).groupby("label_key").s.sum()
            assert (d[d.measure == "FLAG"].groupby("label_key").size().loc[q.index] == N_FM).all()
            pts[(pair, coh)] = q
            out[f"{pair}|{coh}"] = {"n": int(len(q)), "higher": int((q.FLAG > 0).sum())}
    allq = pd.concat(pts.values())
    xl = float(np.ceil(allq.FLAG.abs().max() / 5) * 5)
    ylo, yhi = float(np.floor(allq.BA.min() / 5) * 5), float(np.ceil(allq.BA.max() / 5) * 5)
    lc, gc = 0.105, 0.022
    wc = (right - lc - 2 * gc) / 3
    for k, (pair, who) in enumerate(PAIR_NAMES):
        ax = fig.add_axes([lc + k * (wc + gc), (H - 9.15) / H, wc, 1.95 / H]); style(ax, grid=None)
        ax.grid(True, color=GRID, lw=0.6)
        ax.axhline(0, color=INK2, lw=0.8); ax.axvline(0, color=INK2, lw=0.8)
        for coh, mk in (("EchoNext", "o"), ("MIMIC-IV", "s")):
            for lk, r in pts[(pair, coh)].iterrows():
                ax.plot(r.FLAG, r.BA, mk, ms=5.6 if r.n >= 100 else 3.6, mfc=COH[coh], mec="white", mew=0.7, zorder=3)
                if lk in (E_AS, M_AS):
                    ax.plot(r.FLAG, r.BA, "o", ms=10.5, mfc="none", mec=INK, mew=0.9, zorder=5)
        ax.set_xlim(-xl - 1, xl + 1); ax.set_ylim(ylo - 1, yhi + 1)
        ax.xaxis.set_major_locator(MultipleLocator(5)); ax.yaxis.set_major_locator(MultipleLocator(5))
        if k:
            ax.set_yticklabels([])
        e, m = out[f"{pair}|EchoNext"], out[f"{pair}|MIMIC-IV"]
        word = "higher" if e["higher"] > e["n"] / 2 else "lower"
        cnt = lambda s: s["higher"] if word == "higher" else s["n"] - s["higher"]
        ax.text(0, 1.165, f"{who} − White", transform=ax.transAxes, fontsize=8.4, fontweight="bold", color=INK, va="bottom", ha="left")
        ax.text(0, 1.025, f"flagging rate {word} for {cnt(e)} of {e['n']} diagnoses\n(EchoNext) and {cnt(m)} of {m['n']} (MIMIC-IV)", transform=ax.transAxes,
                fontsize=6.3, color=INK2, va="bottom", ha="left", linespacing=1.15)
        ax.text(0.985, 0.03, "flagged more often →", transform=ax.transAxes, fontsize=6.3, color=MUTED, ha="right", va="bottom")
        if k == 0:
            ax.text(0.02, 0.03, "↓ separated\nless well", transform=ax.transAxes, fontsize=6.3, color=MUTED, ha="left", va="bottom", linespacing=1.05)
            ax.set_ylabel("Difference in balanced accuracy\n(percentage points)", fontsize=7.8, color=INK2)
            ax.annotate("c", xy=(0, 1), xycoords="axes fraction", xytext=(0, 44), textcoords="offset points", fontsize=10.5, fontweight="bold", color=INK,
                        va="bottom", ha="left", annotation_clip=False)
            ax.annotate(f"Every diagnosis: flagging rate and balanced accuracy, mean of the {N_WORD} models", xy=(0, 1), xycoords="axes fraction",
                        xytext=(13, 44.8), textcoords="offset points", fontsize=8.6, fontweight="bold", color=INK, va="bottom", ha="left", annotation_clip=False)
        if k == 1:
            ax.set_xlabel("Difference from White patients in flagging rate (percentage points, same age and sex)", fontsize=7.8, color=INK2)
    h = [Line2D([], [], marker="o", color=CE, lw=0, ms=5.4, label="EchoNext"), Line2D([], [], marker="s", color=CM, lw=0, ms=5.4, label="MIMIC-IV"),
         Line2D([], [], marker="o", mfc="none", mec=INK, lw=0, ms=9.5, mew=0.9, label="Aortic stenosis"),
         Line2D([], [], marker="o", mfc=INK2, mec="white", lw=0, ms=3.6, mew=0.7, label="Small: fewer than 100 patients with the diagnosis")]
    fig.legend(handles=h, loc="lower center", ncol=4, frameon=False, fontsize=6.9, handletextpad=0.3, columnspacing=1.6, bbox_to_anchor=(0.545, 0.004))
    save_figure(fig, "Fig2")
    return out


fig2_stats = figure2(R)

## Fig. 3 | Erasing race information changes flagging rates but not balanced accuracy

**a**, **b**, Black–White difference in flagging rate (**a**) and in balanced accuracy (**b**) at the same age and sex, before (black markers) and after least-squares concept erasure (LEACE, arrowheads). Coloured arrows mark changes whose 95% confidence interval excluded zero. **c**, **d**, Change after LEACE in the difference in flagging rate (**c**) and in balanced accuracy (**d**) against the Black–White difference in prevalence among all ECGs, for each diagnosis and cohort as the mean of the three models, with vertical lines for the range across the three models (not confidence intervals). The dashed lines are least-squares fits, and r is the Pearson correlation coefficient with its two-sided P value. ECG, electrocardiogram; SHD, structural heart disease.

In [ ]:
def figure3(R):
    H = 7.95
    fig = plt.figure(figsize=(7.2, H))
    left, right, gx = 0.178, 0.988, 0.03
    w = (right - left - gx) / 2
    ya, ha = 1 - (0.42 + 2.75) / H, 2.75 / H
    for j, (meas, title) in enumerate((("FLAG", "Flagging rate"), ("BA", "Balanced accuracy"))):
        ax = fig.add_axes([left + j * (w + gx), ya, w, ha])
        ys = block_axis(ax, j, (-15, 10), 5)
        for b, (_, e_lab, m_lab) in enumerate(ROWS):
            for coh, lab, dy in (("EchoNext", e_lab, -0.17), ("MIMIC-IV", m_lab, 0.17)):
                d = tmv(R, coh, lab, meas, eraser="LEACE")
                for fm in ORDER:
                    r, y = d.loc[fm], ys[(b, fm)] + dy
                    sig = (r.change_lo > 0) or (r.change_hi < 0)
                    ax.annotate("", xy=(100 * r.value, y), xytext=(100 * r.before, y),
                                arrowprops=dict(arrowstyle="-|>", color=COH[coh] if sig else MUTED, lw=1.7 if sig else 1.0, shrinkA=0, shrinkB=0, mutation_scale=8))
                    ax.plot(100 * r.before, y, "o" if coh == "EchoNext" else "s", color=INK, ms=4.0, zorder=3)
        ax.set_xlabel(f"Black − White difference in {title.lower()}\n(percentage points, same age and sex)", fontsize=7.8, color=INK2)
        panel(ax, "ab"[j], title)
        hints(ax, "separated less well" if meas == "BA" else None, "flagged more often" if meas == "FLAG" else None)
    h = [Line2D([], [], color=INK, marker="o", ms=4.0, lw=0, label="Before erasure (circle EchoNext, square MIMIC-IV)"),
         Line2D([], [], color=CE, marker=">", ms=4.5, lw=1.7, label="After LEACE, EchoNext"), Line2D([], [], color=CM, marker=">", ms=4.5, lw=1.7, label="After LEACE, MIMIC-IV"),
         Line2D([], [], color=MUTED, lw=1.0, label="Grey: change not distinguishable from zero")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.58, ya - 1.02 / H), ncol=2, frameon=False, fontsize=7.4, handletextpad=0.5, columnspacing=1.6)
    # ---- c, d: prevalence difference among all ECGs against the change after LEACE (one point per diagnosis and cohort; 12 + 7 = 19)
    T = R["tm"]
    T = T[(T.eraser == "LEACE") & (T.pair == "black-white") & T.valid]
    pv = R["prev"].pivot_table(index=["cohort", "label_key"], columns="group", values="prev_all_ecgs")
    G = {}
    for meas in ("FLAG", "BA"):
        g = T[T.measure == meas].groupby(["cohort", "label_key", "label"]).change.agg(["mean", "min", "max", "size"]).reset_index()
        assert (g["size"] == N_FM).all()
        g["x"] = [100 * (pv.loc[(c, l), "black"] - pv.loc[(c, l), "white"]) for c, l in zip(g.cohort, g.label_key)]
        g[["mean", "min", "max"]] *= 100
        G[meas] = g
    F = G["FLAG"]
    x, yv = F.x.to_numpy(), F["mean"].to_numpy()
    rr, pr = (float(v) for v in pearsonr(x, yv))                # two-sided p value
    rb, pb = (float(v) for v in pearsonr(G["BA"].x.to_numpy(), G["BA"]["mean"].to_numpy()))
    lo, hi = min(F["min"].min(), G["BA"]["min"].min()), max(F["max"].max(), G["BA"]["max"].max())
    wc, gc, yc, hc = 0.355, 0.07, 0.98 / H, 2.3 / H
    for j, (meas, title, letter) in enumerate((("FLAG", "Change in the flagging-rate difference", "c"), ("BA", "Change in the balanced-accuracy difference", "d"))):
        g = G[meas]
        ax = fig.add_axes([left + j * (wc + gc), yc, wc, hc]); style(ax, grid="y")
        ax.axhline(0, color=INK2, lw=0.8); ax.axvline(0, color=INK2, lw=0.8)
        gb1, gb0 = np.polyfit(g.x, g["mean"], 1)
        xx = np.array([g.x.min() - 0.8, g.x.max() + 0.8])
        ax.plot(xx, gb0 + gb1 * xx, color=MUTED, lw=1.0, ls="--", zorder=1)
        for coh in ("EchoNext", "MIMIC-IV"):
            d = g[g.cohort == coh]
            ax.vlines(d.x, d["min"], d["max"], color=COH[coh], lw=1.0, alpha=0.7)
            ax.plot(d.x, d["mean"], "o" if coh == "EchoNext" else "s", color=COH[coh], ms=4.8, mec="white", mew=0.6, lw=0)
        a_ = g[g.label == "Aortic stenosis"]
        ax.plot(a_.x, a_["mean"], "o", ms=10, mfc="none", mec=INK, mew=0.9, zorder=4)
        if meas == "FLAG":
            ax.text(a_.x.max() + 0.65, a_["mean"].mean(), "Aortic\nstenosis", fontsize=6.8, color=INK2, ha="left", va="center", linespacing=1.1)
        ax.set_ylim(lo - 0.6, hi + 0.6)
        ax.xaxis.set_major_locator(MultipleLocator(5)); ax.yaxis.set_major_locator(MultipleLocator(2))
        rg, pg = (rr, pr) if meas == "FLAG" else (rb, pb)
        note = f"r = {rg:.2f}, P {pfmt(pg)}\n".replace("-", MINUS) + f"mean absolute change {np.abs(g['mean']).mean():.1f} points"
        ax.text(0.985, 0.975, note, transform=ax.transAxes, fontsize=6.9, color=INK, ha="right", va="top", linespacing=1.25)
        ax.set_xlabel("Black − White difference in prevalence\namong all ECGs (percentage points)", fontsize=7.8, color=INK2)
        if j == 0:
            ax.set_ylabel(f"Change after LEACE, mean of\nthe {N_WORD} models (percentage points)", fontsize=7.8, color=INK2)
        else:
            ax.set_yticklabels([])
        panel(ax, letter, title)
    h = [Line2D([], [], color=CE, marker="o", ms=4.8, lw=0, label="EchoNext"), Line2D([], [], color=CM, marker="s", ms=4.8, lw=0, label="MIMIC-IV"),
         Line2D([], [], marker="o", mfc="none", mec=INK, lw=0, ms=9, mew=0.9, label="Aortic stenosis"),
         Line2D([], [], color=INK2, marker="|", ms=9, mew=1.0, lw=0, label=f"Line: range across the {N_WORD} models")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.56, 0.0), ncol=4, frameon=False, fontsize=7.4, handletextpad=0.3, columnspacing=1.8)
    save_figure(fig, "Fig3")
    return {"r": rr, "p": pr, "r_ba": rb, "p_ba": pb, "n_points": int(len(x)), "n_by_cohort": F.cohort.value_counts().to_dict(),
            "mean_abs_change_flag": float(np.abs(yv).mean()), "mean_abs_change_ba": float(G["BA"]["mean"].abs().mean()), "rho": float(spearmanr(x, yv)[0])}


fig3_stats = figure3(R)
fig3_stats

## Fig. 4 | Why erasure shifts the scores of a group, and what this does to missed patients and false-positive results

**a**, Illustration with made-up numbers. Each dot is the ECG of one patient. Black and White patients differ in features unrelated to disease (1), and more Black patients have the disease, which moves the Black average towards the disease (2). Standard erasure, fitted on all ECGs, removes both parts, so the scores of Black patients fall and more of those with the disease end up below the threshold. Disease-negative erasure, fitted on ECGs without the disease, removes only the first part and leaves the scores unchanged. Scores are drawn as in Fig. 2a, with patients with the disease above the axis, those without below it and White patients as the dashed reference. **b**, **c**, Change after erasure in the Black–White difference in the false-negative rate (FNR, **b**) and in the false-positive rate (FPR, **c**) at the same age and sex, for standard erasure (filled markers) and disease-negative erasure (open markers), as the mean of the three models. Horizontal lines show the range across the three models (not confidence intervals), hidden by the marker when it is very short. ECG, electrocardiogram; SHD, structural heart disease.

In [ ]:
def figure4(R):
    MK = {"EchoNext": "o", "MIMIC-IV": "s"}
    LIGHT_F, LIGHT_E, DARK4 = "#d4d2ca", "#8f8d86", INK2
    PW, PB, SDS = 0.40, 0.60, 0.45                                            # illustrative shares with the disease; spread of the scores
    THR = 1.0 - norm.ppf(0.80) * SDS                                          # gives White patients 80% sensitivity
    BADGE, BS = dict(boxstyle="circle,pad=0.18", fc="white", ec=INK2, lw=0.8), 6.8
    H = 6.67
    fig = plt.figure(figsize=(7.2, H), dpi=400)                               # laid out at the saving resolution (numbers in circles inside text lines)

    def badge(ax, x, y, s, **kw):
        return ax.text(x, y, s, fontsize=BS, fontweight="bold", color=INK, ha="center", va="center", bbox=BADGE, **kw)

    def rich(ax, x, y, parts, size=6.5, color=INK2, data=False):
        """A line of text with numbers in circles ('b' parts) between words ('t' parts)."""
        r = fig.canvas.get_renderer()
        if data:
            x, y = ax.transAxes.inverted().transform(ax.transData.transform((x, y)))
        to_px, to_ax = ax.transAxes.transform, ax.transAxes.inverted().transform
        px, space = to_px((x, y))[0], 0.28 * size * fig.dpi / 72
        for i, (kind, s) in enumerate(parts):
            if kind == "t":
                if i and s[:1] not in ",.;:":
                    px += space
                px = ax.text(to_ax((px, 0))[0], y, s, transform=ax.transAxes, fontsize=size, color=color, ha="left", va="center").get_window_extent(r).x1
            else:
                probe = ax.text(0, 0, s, fontsize=BS, fontweight="bold")
                e = probe.get_window_extent(r); probe.remove()
                d = max(e.width, e.height) + 2 * 0.18 * BS * fig.dpi / 72
                px += space
                badge(ax, to_ax((px + d / 2, 0))[0], y, s, transform=ax.transAxes)
                px += d

    def cluster(n, cx, cy, r=0.085):
        pts, ring = [(cx, cy)], 1
        while len(pts) < n:
            k = 6 * ring
            for i in range(k):
                if len(pts) == n:
                    break
                a_ = 2 * np.pi * i / k + ring * 0.3
                pts.append((cx + ring * r * np.cos(a_), cy + ring * r * 0.95 * np.sin(a_)))
            ring += 1
        return np.array(pts)

    # ---- a, left: the ECGs of the two groups and the two parts of the difference between their averages
    top, ha_ = 0.36, 2.45
    ax = fig.add_axes([0.075, 1 - (top + ha_) / H, 0.43, ha_ / H]); ax.axis("off")
    ax.set_xlim(-0.62, 1.95); ax.set_ylim(-0.66, 1.62)
    ax.annotate("", xy=(1.42, -0.42), xytext=(-0.4, -0.42), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1.0, shrinkA=0, shrinkB=0))
    ax.annotate("", xy=(-0.4, 1.42), xytext=(-0.4, -0.42), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1.0, shrinkA=0, shrinkB=0))
    ax.text(0.5, -0.5, "ECG features of the disease (read by the detector)", fontsize=6.8, color=INK2, ha="center", va="top")
    ax.text(-0.46, 0.5, "ECG features that differ\nbetween groups", fontsize=6.8, color=INK2, ha="right", va="center", rotation=90, linespacing=1.1)
    for y, p_, face, edge in ((0.0, PW, LIGHT_F, LIGHT_E), (1.0, PB, DARK4, DARK4)):
        k = int(round(p_ * 20))
        for cx, m in ((0.0, 20 - k), (1.0, k)):
            xy = cluster(m, cx, y)
            ax.scatter(xy[:, 0], xy[:, 1], s=13, c=face, edgecolors=edge, linewidths=0.6, zorder=4)
    ax.text(0.0, 1.25, "without\nthe disease", fontsize=6.6, color=INK2, ha="center", va="bottom", linespacing=1.05)
    ax.text(1.0, 1.25, "with\nthe disease", fontsize=6.6, color=INK2, ha="center", va="bottom", linespacing=1.05)
    ax.text(1.3, 1.0, f"Black patients\ne.g. {100 * PB:.0f}% with the disease", fontsize=6.7, color=INK, ha="left", va="center", linespacing=1.15)
    ax.text(1.3, 0.0, f"White patients\ne.g. {100 * PW:.0f}% with the disease", fontsize=6.7, color=INK, ha="left", va="center", linespacing=1.15)
    wa, ba = (PW, 0.0), (PB, 1.0)
    ax.plot(*wa, "D", ms=6.5, mfc=LIGHT_F, mec=LIGHT_E, mew=1.0, zorder=6)
    ax.plot(*ba, "D", ms=6.5, mfc=DARK4, mec=DARK4, mew=1.0, zorder=6)
    ax.text(PW, -0.17, "average", fontsize=6.3, color=INK2, ha="center", va="top")
    ax.text(PB, 1.17, "average", fontsize=6.3, color=INK2, ha="center", va="bottom")
    ax.annotate("", xy=ba, xytext=wa, arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.4, shrinkA=5, shrinkB=5, mutation_scale=9), zorder=5)
    ax.annotate("", xy=(PW, 0.97), xytext=wa, arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.0, ls=(0, (2, 1.6)), shrinkA=5, shrinkB=2, mutation_scale=7), zorder=3)
    ax.annotate("", xy=(PB - 0.01, 0.84), xytext=(PW, 0.84), arrowprops=dict(arrowstyle="-|>", color=INK2, lw=1.0, ls=(0, (2, 1.6)), shrinkA=0, shrinkB=0,
                                                                         mutation_scale=7), zorder=3)
    badge(ax, PW - 0.07, 0.45, "1", zorder=7)
    ax.text(PW - 0.34, 0.45, "differs\nregardless\nof disease", fontsize=6.3, color=INK2, ha="center", va="center", linespacing=1.05)
    badge(ax, 0.5, 0.72, "2", zorder=7)
    ax.text(0.6, 0.56, "more Black patients\nhave the disease", fontsize=6.3, color=INK2, ha="left", va="center", linespacing=1.05)
    ax.text(0.6, 0.33, "difference between", fontsize=6.3, color=INK, ha="left", va="center")
    rich(ax, 0.6, 0.2, [("t", "the averages ="), ("b", "1"), ("t", "+"), ("b", "2")], size=6.3, color=INK, data=True)
    ax.annotate("a", xy=(0, 1), xycoords="axes fraction", xytext=(-12, 2), textcoords="offset points", fontsize=10.5, fontweight="bold", color=INK, va="bottom")
    ax.annotate("Why erasure shifts the scores of a group (illustration)", xy=(0, 1), xycoords="axes fraction", xytext=(2, 2.8), textcoords="offset points",
                fontsize=8.6, fontweight="bold", color=INK, va="bottom")
    ax.annotate("All numbers in panel a are made-up examples, not study results.", xy=(0, 1), xycoords="axes fraction", xytext=(2, -2),
                textcoords="offset points", fontsize=6.6, fontstyle="italic", color=MUTED, va="top")

    # ---- a, right: detector scores after each form of erasure, drawn as in Fig. 2a
    def scores(rect, title, sub, shift):
        bx = fig.add_axes(rect); bx.axis("off")
        xs = np.linspace(-1.35, 2.2, 600)
        hgt = 0.85 / norm.pdf(0, 0, SDS)
        bx.set_xlim(-1.75, 2.75); bx.set_ylim(-1.15, 2.05)
        bx.plot([-1.35, 2.2], [0, 0], color=AXIS, lw=1.0)
        bx.annotate("", xy=(2.25, 0), xytext=(2.1, 0), arrowprops=dict(arrowstyle="-|>", color=AXIS, lw=1.0, shrinkA=0, shrinkB=0))
        bx.text(2.3, 0, "detector\nscore", fontsize=6.2, color=MUTED, ha="left", va="center", linespacing=1.0)
        for m, sgn in ((1.0, 1), (0.0, -1)):
            bx.plot(xs, sgn * hgt * norm.pdf(xs, m, SDS), color=LIGHT_E, lw=1.1, ls=(0, (3, 1.6)), zorder=3)
        for m, sgn in ((1.0 - shift, 1), (0.0 - shift, -1)):
            y = sgn * hgt * norm.pdf(xs, m, SDS)
            bx.plot(xs, y, color=DARK4, lw=1.4, zorder=4)
            bx.fill_between(xs, 0, y, where=(xs < THR) if sgn > 0 else (xs > THR), color=DARK4, alpha=0.28, lw=0, zorder=2)
        bx.plot([THR, THR], [-1.0, 1.08], color=INK, lw=0.9, ls=(0, (4, 2)), zorder=5)
        bx.text(THR - 0.06, 1.08, "threshold", fontsize=6.2, color=INK, ha="right", va="top")
        bx.text(-1.35, 0.42, "with the\ndisease", fontsize=6.2, color=INK2, ha="left", va="center", linespacing=1.0)
        bx.text(-1.35, -0.42, "without", fontsize=6.2, color=INK2, ha="left", va="center")
        sb, fb = 1 - norm.cdf(THR, 1.0 - shift, SDS), 1 - norm.cdf(THR, 0.0 - shift, SDS)
        sw, fw = 1 - norm.cdf(THR, 1.0, SDS), 1 - norm.cdf(THR, 0.0, SDS)
        bx.text(2.75, -0.42, "example", fontsize=6.1, fontstyle="italic", color=MUTED, ha="right", va="center")
        bx.text(2.75, -0.8, f"missed: Black {100 * (1 - sb):.0f}%, White {100 * (1 - sw):.0f}%\nfalse positives: Black {100 * fb:.0f}%, White {100 * fw:.0f}%",
                fontsize=6.1, color=INK, ha="right", va="center", linespacing=1.2)
        bx.text(-1.75, 2.05, title, fontsize=7.4, fontweight="bold", color=INK, ha="left", va="top")
        rich(bx, 0.0, 0.855, sub)

    hl = 1.12
    scores([0.53, 1 - (top - 0.1 + hl) / H, 0.46, hl / H], "After standard erasure (fitted on all ECGs)",
           [("t", "removes"), ("b", "1"), ("t", "and"), ("b", "2"), ("t", ", so Black scores fall by part"), ("b", "2")], PB - PW)
    scores([0.53, 1 - (top + 2 * hl) / H, 0.46, hl / H], "After disease-negative erasure (fitted on ECGs without the disease)",
           [("t", "removes"), ("b", "1"), ("t", "only, so Black scores do not change")], 0.0)
    kx = fig.add_axes([0.53, 1 - (top + 0.12 + 2 * hl + 0.22) / H, 0.46, 0.2 / H]); kx.axis("off"); kx.set_xlim(0, 1); kx.set_ylim(0, 1)
    kx.plot([0.02, 0.08], [0.5, 0.5], color=DARK4, lw=1.4); kx.text(0.095, 0.5, "Black patients", fontsize=6.4, color=INK2, va="center")
    kx.plot([0.32, 0.38], [0.5, 0.5], color=LIGHT_E, lw=1.1, ls=(0, (3, 1.6))); kx.text(0.395, 0.5, "White patients", fontsize=6.4, color=INK2, va="center")
    kx.fill_between([0.62, 0.68], 0.25, 0.75, color=DARK4, alpha=0.28, lw=0); kx.text(0.695, 0.5, "missed or false positive (Black)", fontsize=6.4, color=INK2, va="center")

    # ---- b, c: missed patients and false-positive results after standard and disease-negative erasure
    T = R["tm"]
    T = T[(T.pair == "black-white") & T.valid]
    blocks = [("Composite SHD", {"EchoNext": E_COMP, "MIMIC-IV": M_COMP}, "LEACE (SHD-neg fit)"),
              ("Aortic stenosis", {"EchoNext": E_AS, "MIMIC-IV": M_AS}, "LEACE (AS-neg fit)")]
    ys, heads, y = {}, [], 0.0
    for b, (name, _, _) in enumerate(blocks):
        heads.append((y - 0.8, name))
        for coh in ("EchoNext", "MIMIC-IV"):
            ys[(b, coh)] = y; y += 1
        y += 0.75
    ymax = y - 0.75 - 0.45
    yb, hb = 1.07 / H, 2.3 / H
    left, right, gx = 0.13, 0.988, 0.04
    w = (right - left - gx) / 2
    out = {}
    for j, (meas, title, xl, hint) in enumerate((("FNR", "Missed patients", "missed patients (FNR)", "missed more often"),
                                                ("FPR", "False-positive results", "false-positive results (FPR)", "more false positives"))):
        ax = fig.add_axes([left + j * (w + gx), yb, w, hb]); style(ax)
        ax.axvline(0, color=INK2, lw=0.8)
        for (b, coh), yy in ys.items():
            if coh == "EchoNext":
                ax.axhline(yy + 0.5, color=GRID, lw=0.6, zorder=0)
        for b, (name, labs, neg) in enumerate(blocks):
            for coh in ("EchoNext", "MIMIC-IV"):
                for er, dy, filled in (("LEACE", -0.17, True), (neg, 0.17, False)):
                    d = T[(T.cohort == coh) & (T.label_key == labs[coh]) & (T.eraser == er) & (T.measure == meas)]
                    assert len(d) == N_FM, (coh, name, er, meas)
                    v = 100 * d.change
                    out[f"{name}|{coh}|{er}|{meas}"] = [float(v.mean()), float(v.min()), float(v.max())]
                    yy = ys[(b, coh)] + dy
                    ax.hlines(yy, v.min(), v.max(), color=COH[coh], lw=1.3, alpha=0.8)
                    ax.plot(v.mean(), yy, MK[coh], ms=5.6, lw=0, mfc=COH[coh] if filled else "white", mec=COH[coh], mew=1.2, zorder=4)
        ax.set_yticks(list(ys.values())); ax.set_yticklabels([c for (_, c) in ys] if j == 0 else [], fontsize=7.8, color=INK)
        ax.set_ylim(ymax, -1.3); ax.set_xlim(-10, 10); ax.xaxis.set_major_locator(MultipleLocator(5))
        if j == 0:
            for yy, name in heads:
                ax.text(-9.8, yy, name, fontsize=8.0, fontweight="bold", color=INK, va="center", ha="left")
        ax.set_xlabel(f"Change in the Black − White difference in\n{xl}, percentage points\n(mean of the {N_WORD} models; line: range)", fontsize=7.6, color=INK2)
        panel(ax, "bc"[j], title)
        hints(ax, None, hint)
    h = [Line2D([], [], color=INK2, marker="o", ms=5.6, lw=0, label="Standard erasure"),
         Line2D([], [], color=INK2, marker="o", ms=5.6, lw=0, mfc="white", mew=1.2, label="Disease-negative erasure"),
         Line2D([], [], color=CE, marker="o", ms=5.6, lw=1.3, label="EchoNext"), Line2D([], [], color=CM, marker="s", ms=5.6, lw=1.3, label="MIMIC-IV")]
    fig.legend(handles=h, loc="lower center", bbox_to_anchor=(0.53, 0.0), ncol=4, frameon=False, fontsize=7.0, handletextpad=0.3, columnspacing=1.2)
    save_figure(fig, "Fig4")
    return out


fig4_stats = figure4(R)

## Table 2 | Study questions and summary of results

AUROC, area under the receiver operating characteristic curve; ECG, electrocardiogram; FNR, false-negative rate; LEACE, least-squares concept erasure; SHD, structural heart disease.

Values are ranges across the three models, Black minus White, in percentage points for questions 3 to 7, where all differences are standardised for age and sex. Parentheses give the models, of three, in which the answer held in EchoNext and in MIMIC-IV, judged by 95% confidence intervals: both gains above zero (question 1), AUROC above the structure-only model in both analyses (question 2), not below zero (composite SHD in question 4) or excluding zero in the stated direction (all other answers). Flagging rate is the mean of sensitivity and false-positive rate, and balanced accuracy the mean of sensitivity and specificity.

In [ ]:
def ranges(R):
    """The numbers of Table 2 (ranges across the three models and the number of models in which an answer holds)."""
    n = {}
    for k, s1, cmp_, S, c, comp, as_ in (("e", R["s1e"], R["s1ec"], R["e2"], "EchoNext", E_COMP, E_AS), ("m", R["s1m"], R["s1mc"], R["m2"], "MIMIC-IV", M_COMP, M_AS)):
        bw = [s1.loc[KEY[f], "bw_auroc"] for f in ORDER]
        n[f"{k}_bw"] = rngd(min(bw), max(bw))
        ce = cmp_[(cmp_.metric == "bw") & (cmp_.comparison == "FM vs random weights")].set_index("a")
        me = cmp_[(cmp_.metric == "bw") & (cmp_.comparison == "FM vs ECG measurements")].set_index("a")
        n[f"{k}_gain_rand"], n[f"{k}_gain_meas"] = (ce.delta.min(), ce.delta.max()), (me.delta.min(), me.delta.max())
        n[f"{k}_q1"] = int(((ce.delta_lo > 0) & (me.delta_lo > 0)).sum())
        so = S["m0"].loc["structure only", "bw_auroc"]
        Cr = S["C"][S["C"].variant == "removed"].set_index("model")
        noshd = S["A"].loc[list(FMS), "bw_auroc"]
        n[f"{k}_noshd"], n[f"{k}_resid"], n[f"{k}_so"] = rngd(noshd.min(), noshd.max()), (Cr.delta.min(), Cr.delta.max()), so
        n[f"{k}_q2"] = int(sum(S["A"].loc[f, "bw_auroc"] > so and Cr.loc[f, "bw_auroc"] > so for f in FMS))
        for tag, lab in (("c", comp), ("a", as_)):
            F, Bq = tmq(R, c, lab, "FLAG"), tmq(R, c, lab, "BA")
            Fe, Be = tmq(R, c, lab, "FLAG", eraser="LEACE"), tmq(R, c, lab, "BA", eraser="LEACE")
            n[f"{k}_{tag}Flag"], n[f"{k}_{tag}BA"] = pp(F.value.min(), F.value.max()), pp(Bq.value.min(), Bq.value.max())
            n[f"{k}_{tag}Flag_up"], n[f"{k}_{tag}Flag_dn"] = int((F.lo > 0).sum()), int((F.hi < 0).sum())
            n[f"{k}_{tag}BA_dn"] = int((Bq.hi < 0).sum())
            n[f"{k}_{tag}Flag_ch"] = pp(Fe.change.min(), Fe.change.max())
            n[f"{k}_{tag}Flag_ch_up"], n[f"{k}_{tag}Flag_ch_dn"] = int((Fe.change_lo > 0).sum()), int((Fe.change_hi < 0).sum())
            n[f"{k}_{tag}BA_ch"] = (Be.change.min(), Be.change.max())
            n[f"{k}_{tag}BA_ch_sig"] = int(((Be.change_lo > 0) | (Be.change_hi < 0)).sum())
            fn = tmq(R, c, lab, "FNR", eraser="LEACE")
            n[f"{k}_{tag}FNR_ch"] = pp(fn.change.min(), fn.change.max())
            n[f"{k}_{tag}FNR_ch_up"], n[f"{k}_{tag}FNR_ch_dn"] = int((fn.change_lo > 0).sum()), int((fn.change_hi < 0).sum())
        lo = min(n[f"{k}_cBA_ch"][0], n[f"{k}_aBA_ch"][0]); hi = max(n[f"{k}_cBA_ch"][1], n[f"{k}_aBA_ch"][1])
        n[f"{k}_BA_ch"] = f"{signed(round(100 * lo, 1))} to {signed(round(100 * hi, 1))}"
        x2 = R["x2"]
        b = x2[(x2.cohort == c) & (x2.pair == "black-white") & (x2.measure == "FLAG") & (x2.variant == "equalised")].set_index("fm").loc[ORDER]
        n[f"{k}_x2"], n[f"{k}_x2_up"] = pp(b["diff"].min(), b["diff"].max()), int((b.lo > 0).sum())
    return n


def d3(lohi):
    """Range of signed changes with three decimals; one value when both ends print alike."""
    a, b = (f"{v:+.3f}".replace("-", MINUS) for v in lohi)
    return a if a == b else f"{a} to {b}"


n = ranges(R)
k = lambda a, b: f"{a}/{N_FM}; {b}/{N_FM}"
rows = [
    ["1. Can Black and White patients be distinguished from foundation model embeddings, and better than from ECG measurements or random-weight encoders?",
     "AUROC of linear probe (gain over random weights; gain over ECG measurements)",
     f"{n['e_bw']} ({d3(n['e_gain_rand'])}; {d3(n['e_gain_meas'])})", f"{n['m_bw']} ({d3(n['m_gain_rand'])}; {d3(n['m_gain_meas'])})", f"Yes ({k(n['e_q1'], n['m_q1'])})"],
    ["2. Does this hold among patients without recorded SHD and after adjustment for cardiac structure?",
     "AUROC without recorded SHD; change in AUROC after removal of structural components (AUROC of structure-only model)",
     f"{n['e_noshd']}; {d3(n['e_resid'])} ({n['e_so']:.2f})", f"{n['m_noshd']}; {d3(n['m_resid'])} ({n['m_so']:.2f})", f"Yes ({k(n['e_q2'], n['m_q2'])})"],
    ["3. At the same age and sex, are Black patients flagged at a different rate (shifted operating point)?",
     "Difference in flagging rate: composite SHD; aortic stenosis", f"{n['e_cFlag']}; {n['e_aFlag']}", f"{n['m_cFlag']}; {n['m_aFlag']}",
     f"Composite SHD: more often ({k(n['e_cFlag_up'], n['m_cFlag_up'])}). Aortic stenosis: less often ({k(n['e_aFlag_dn'], n['m_aFlag_dn'])})"],
    ["4. At the same age and sex, is disease separated less well in Black patients?", "Difference in balanced accuracy: composite SHD; aortic stenosis",
     f"{n['e_cBA']}; {n['e_aBA']}", f"{n['m_cBA']}; {n['m_aBA']}",
     f"Composite SHD: no ({k(N_FM - n['e_cBA_dn'], N_FM - n['m_cBA_dn'])}). Aortic stenosis: yes ({k(n['e_aBA_dn'], n['m_aBA_dn'])})"],
    ["5. Is the higher flagging of Black patients for composite SHD explained by the prevalence of SHD in the fitting data?",
     "Difference in flagging rate for composite SHD after prevalence was equalised across groups in the fitting data", n["e_x2"], n["m_x2"],
     f"Partly. Still higher in EchoNext ({n['e_x2_up']}/{N_FM}), largely removed in MIMIC-IV (still higher in {n['m_x2_up']}/{N_FM})"],
    ["6. Does erasure of race information (LEACE) remove these differences?",
     "Change in the difference in flagging rate: composite SHD; aortic stenosis. Change in the difference in balanced accuracy: both diagnoses",
     f"{n['e_cFlag_ch']}; {n['e_aFlag_ch']}. {n['e_BA_ch']}", f"{n['m_cFlag_ch']}; {n['m_aFlag_ch']}. {n['m_BA_ch']}",
     f"No. Flagging rates moved: composite SHD down ({k(n['e_cFlag_ch_dn'], n['m_cFlag_ch_dn'])}), aortic stenosis up ({k(n['e_aFlag_ch_up'], n['m_aFlag_ch_up'])}). "
     f"Balanced accuracy changed in {n['e_cBA_ch_sig'] + n['m_cBA_ch_sig'] + n['e_aBA_ch_sig'] + n['m_aBA_ch_sig']} of 12 comparisons"],
    ["7. What happens to missed patients after erasure?", "Change in the FNR difference: composite SHD; aortic stenosis",
     f"{n['e_cFNR_ch']}; {n['e_aFNR_ch']}", f"{n['m_cFNR_ch']}; {n['m_aFNR_ch']}",
     f"Composite SHD: Black patients missed more often ({k(n['e_cFNR_ch_up'], n['m_cFNR_ch_up'])}). Aortic stenosis: gap narrowed "
     f"({k(n['e_aFNR_ch_dn'], n['m_aFNR_ch_dn'])}), with more false-positive results in Black patients"],
    ["Taken together. The embeddings carry race information that cardiac structure does not explain (questions 1 and 2). Detectors built on them treat "
     "groups differently in two ways. Groups are flagged at different rates although disease is separated equally well, as for composite SHD, and disease "
     "is separated less well in one group, as for aortic stenosis (questions 3 and 4). Prevalence in the fitting data explains part of the first difference "
     "(question 5). Erasing race information moves the operating point and leaves separation unchanged, so that it exchanges missed patients for "
     "unnecessary referrals (questions 6 and 7).", "", "", "", ""],
]
table2 = save_table(pd.DataFrame(rows, columns=["Question", "Measure", "EchoNext", "MIMIC-IV", "Answer (models, EchoNext; MIMIC-IV)"]), "Table2")
with pd.option_context("display.max_colwidth", 200):
    display(table2)

## Numbers quoted in the text that are not shown in a table or figure

In [ ]:
chx = R["ch"].set_index(["cohort", "group"])
x2rows = R["x2rows"]
kept = x2rows.groupby(["cohort", "variant"]).rows.mean()
ra = R["race_after"]
text_numbers = {
    "Race AUROC after LEACE (range over models and cohorts)": rngd(ra.race_bw_auroc.min(), ra.race_bw_auroc.max()),
    "Share of Black patients with composite SHD who had aortic stenosis (EchoNext; MIMIC-IV)":
        "; ".join(f"{100 * chx.loc[(c, 'black'), 'pos_Aortic stenosis'] / chx.loc[(c, 'black'), 'pos_Composite SHD']:.0f}%" for c in ("EchoNext", "MIMIC-IV")),
    "Mean absolute change after LEACE across the 19 diagnosis-cohort combinations: flagging rate; balanced accuracy":
        f"{fig3_stats['mean_abs_change_flag']:.1f}; {fig3_stats['mean_abs_change_ba']:.1f} percentage points",
    "Pearson r (two-sided p): prevalence difference vs change in flagging rate; vs change in balanced accuracy":
        f"{fig3_stats['r']:.2f} (p={fig3_stats['p']:.2g}); {fig3_stats['r_ba']:.2f} (p={fig3_stats['p_ba']:.2g})",
    "Fitting ECGs kept by the prevalence-equalised subsample (EchoNext; MIMIC-IV)":
        "; ".join(f"{100 * kept[(c, 'equalised')] / kept[(c, 'all rows')]:.0f}%" for c in ("EchoNext", "MIMIC-IV")),
    "Median (IQR) days from the most recent ECG to its TTE, MIMIC-IV": chx.loc[("MIMIC-IV", "All"), "gap_days"],
    "Matched Black-White pairs (eligible Black patients); largest |SMD| after matching":
        "; ".join(f"{c}: {R[k_]['sum']['matching']['n_pairs']:,} ({R[k_]['sum']['matching']['n_a_eligible']:,}); {R[k_]['sum']['matching']['max_abs_smd_after']:.2f}"
                  for c, k_ in (("EchoNext", "e2"), ("MIMIC-IV", "m2"))),
}
pd.Series(text_numbers, name="value").to_frame()